In [1]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd

In [ ]:

df = pd.read_parquet(r"..\..\artifacts\validation\valid\credit_approval_valid.parquet")
df.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorID               10000 non-null  int64  
 1   tpep_pickup_datetime   10000 non-null  object 
 2   tpep_dropoff_datetime  10000 non-null  object 
 3   passenger_count        10000 non-null  float64
 4   trip_distance          10000 non-null  float64
 5   RatecodeID             10000 non-null  float64
 6   store_and_fwd_flag     10000 non-null  object 
 7   PULocationID           10000 non-null  int64  
 8   DOLocationID           10000 non-null  int64  
 9   payment_type           10000 non-null  int64  
 10  fare_amount            10000 non-null  float64
 11  extra                  10000 non-null  float64
 12  mta_tax                10000 non-null  float64
 13  tip_amount             10000 non-null  float64
 14  tolls_amount           10000 non-null  float64
 15  imp

In [38]:
df.drop(
    ["tpep_pickup_datetime", "tpep_dropoff_datetime", "cbd_congestion_fee"],
    axis=1,
    inplace=True,
)

In [ ]:
y = df["total_amount"]
X = df.drop(columns=["total_amount"])

cat_cols = ["VendorID", "store_and_fwd_flag"]
num_cols = X.columns.drop(["VendorID", "store_and_fwd_flag"]).tolist()


num_imputer = SimpleImputer(strategy="mean")
num_scaler = StandardScaler()
num_tr = Pipeline([("impute", num_imputer), ("scale", num_scaler)])

cat_imputer = SimpleImputer(strategy="most_frequent")
ohe = OneHotEncoder()
cat_tr = Pipeline(
    [
        ("impute", cat_imputer),
        ("encode", ohe),
    ]
)

processor = ColumnTransformer(
    [("num", num_tr, num_cols), ("cat", cat_tr, cat_cols)],
    remainder="passthrough",
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_processed = processor.fit_transform(X_train)

X_test_processed = processor.transform(X_test)

In [ ]:
import boto3
import os
# os.chdir("../../")

def push_file_to_s3():
    client = boto3.client("s3")
    client.upload_file(
        "pyproject.toml",
        Bucket="credit-approval-285515884731-eu-north-1-an",
        Key="pyproject.toml",
    )


push_file_to_s3()